In [ ]:
import pandas as pd
from faker import Faker
import numpy as np

# 初始化中文数据生成器
fake = Faker('zh_CN')
np.random.seed(2025)  # 固定随机种子保证可重复性


def generate_scores(num, mu=90, sigma=20):
    scores = np.random.normal(mu, sigma, num)
    return np.clip(np.round(scores), 0, 150).astype(int)


# 生成100条记录
data = []
for i in range(1, 101):
    record = [
        f"{i + 999:04d}",  # 学号
        fake.name(),  # 姓名
        *generate_scores(3)  # 三科成绩
    ]
    data.append(record)

# 创建数据框
df = pd.DataFrame(data, columns=["学号", "姓名", "语文", "数学", "英语"])

# 数据校验
assert df[['语文', '数学', '英语']].max().max() <= 150, "分数超过上限"

# 导出Excel
df.to_excel(
    "students_scores.xlsx",
    index=False,
    engine='openpyxl',  # 支持xlsx格式
    sheet_name='2025级成绩')
print("数据生成完成！")

In [ ]:
# 任务1
# 用Pandas读取studentScores.csv，将缺失值丢弃处理导出为新文件studentScoresP.csv，并查看前三行和后两行。
import pandas as pd

# 读取CSV文件（假设文件编码为UTF-8，若报错可尝试encoding='gbk'）
df = pd.read_csv('studentScores.csv')

# 丢弃包含缺失值的行
df_cleaned = df.dropna()

# 导出新文件
df_cleaned.to_csv('studentScoresP.csv', index=False, encoding='utf_8_sig')  # 保证中文兼容性

# 查看前三行和后两行
print("前三行：")
print(df_cleaned.head(3))
print("\n后两行：")
print(df_cleaned.tail(2))

In [1]:
# 任务2
# 选择预处理后的studentScoreP.csv文件中的列姓名、考号、班级、语文、数学、英语，导出为新的csv文件studentScoreP_new.csv；重新读取新的数据集studentScoreP_new.csv，并选择语文、数学、英语介于100分到150分之间的所有数据集，导出为新的文本文件studentScoreP_newGood.txt，要求数据之间用逗号分隔，每行末尾包含换行符。

import pandas as pd

# 1. 选择指定列并导出为新CSV
df = pd.read_csv('studentScoresP.csv')
df_selected = df[['姓名', '考号', '班级', '语文', '数学', '英语']]
df_selected.to_csv('studentScoresP_new.csv', index=False, encoding='utf_8_sig')

# 2. 重新读取新文件并筛选条件
df_new = pd.read_csv('studentScoresP_new.csv')
condition = ((df_new['语文'].between(100, 150)) & (df_new['数学'].between(100, 150)) & (df_new['英语'].between(100, 150)))
filtered_df = df_new[condition]

# 3. 导出为文本文件（逗号分隔+换行符）
filtered_df.to_csv('studentScoresP_newGood.txt', sep=',', index=False, lineterminator='\n')

# 验证结果
print("筛选后的数据行数:", len(filtered_df))
print("示例数据（前两行）：")
print(filtered_df.head(2).to_string(index=False))

筛选后的数据行数: 3
示例数据（前两行）：
 姓名   考号 班级    语文    数学    英语
郭凤英 1015 二班 121.0 114.0 109.0
 李霞 1076 一班 125.0 110.0 105.0


In [ ]:
# 任务3
# 重新读取studentScoresP_new.csv，按照列班级分类汇总各班语文、数学、英语的平均成绩，并将分组计算结果导出到文本文件studentScoresP_MeanGroup.txt中，要求分组名不作为列名。
import pandas as pd

# 读取新文件（注意文件名实际应为studentScoresP_new.csv，根据真实文件名调整）
df = pd.read_csv('studentScoresP_new.csv')

# 按班级分组并计算各科平均分
group_mean = df.groupby('班级')[['语文', '数学', '英语']].mean()

# 导出为文本文件，班级作为索引且不显示列名
group_mean.to_csv(
    'studentScoreP_MeanGroup.txt',
    header=False,    # 不保留列名
    index=True,      # 保留班级作为索引
    sep=',',         # 逗号分隔
    float_format='%.2f'  # 保留两位小数（可选）
)

# 验证结果
print("班级平均分统计：")
print(group_mean.round(2))

班级平均分统计：
       语文     数学     英语
班级                     
一班  89.51  92.95  90.44
二班  85.45  87.55  91.68


In [ ]:
# 任务4
# 重新读取studentScoresP_new.csv，计算每位同学语文、数学、英语的平均成绩，并将平均成绩作为一个新的列“均值”添加到原始文件，并按照列均值降序排列，并将排序后的结果转存到studentScoresP_Mean.xlsx中。
import pandas as pd

# 读取新文件
df = pd.read_csv('studentScoresP_new.csv')

# 计算每位学生的三科均值（按行计算）
df['均值'] = df[['语文', '数学', '英语']].mean(axis=1).round(2)  # 保留两位小数

# 按均值降序排序
df_sorted = df.sort_values(by='均值', ascending=False)

# 导出到Excel文件（需安装openpyxl库）
df_sorted.to_excel('studentScoresP_Mean.xlsx', index=False)

# 验证结果（打印前3行）
print("排序后数据示例：")
print(df_sorted.head(3).to_string(index=False))

排序后数据示例：
 姓名   考号 班级    语文    数学    英语     均值
郭凤英 1015 二班 121.0 114.0 109.0 114.67
 李帅 1084 一班 121.0 126.0  97.0 114.67
 李霞 1076 一班 125.0 110.0 105.0 113.33


In [6]:
# 任务5
# 读取新的数据集studentScoresP_Mean.xlsx，统计字段均值的最大值maxValue，四分之三位数threeQuartersValue，中位数medianValue，四分之一位数quarterValue，最小值minValue。category = [minValue, quarterValue, medianValue, threeQuartersValue, maxValue]和labels = ['Poor', 'Moderate', 'Good', 'Excellent']将均值进行离散化，并根据离散化结果进行直方图统计，分别画出统计结果的柱状图和饼状图，并分别将柱状图和饼状图保存为studentScoresP_Mean_bar.png，studentScoresP_Mean_pie.png，要求柱状图的x轴刻度以及饼状图均显示labels，图像分辨率不低于300dpi。

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取Excel文件
df = pd.read_excel('studentScoresP_Mean.xlsx')

# 计算统计指标
maxValue = df['均值'].max()
threeQuartersValue = df['均值'].quantile(0.75)
medianValue = df['均值'].median()
quarterValue = df['均值'].quantile(0.25)
minValue = df['均值'].min()

# 定义离散化区间和标签
category = [minValue, quarterValue, medianValue, threeQuartersValue, maxValue]
labels = ['Poor', 'Moderate', 'Good', 'Excellent']

# 对均值进行离散化
df['等级'] = pd.cut(
    df['均值'],
    bins=category,
    labels=labels,
    include_lowest=True  # 确保包含最小值
)

# 统计各等级数量
counts = df['等级'].value_counts().reindex(labels, fill_value=0)  # 确保顺序

# 设置全局字体和图像尺寸
plt.rcParams['font.sans-serif'] = ['SimHei']  # 解决中文显示问题
plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题

# 绘制柱状图
plt.figure(figsize=(10, 6), dpi=300)
bars = plt.bar(labels, counts, color=['#ff9999','#66b3ff','#99ff99','#ffcc99'])
plt.title('均值等级分布柱状图')
plt.xlabel('等级')
plt.ylabel('人数')
# 在柱子上方显示数值
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}',
             ha='center', va='bottom')
plt.savefig('studentScoresP_Mean_bar.png', dpi=300)
plt.close()

# 绘制饼状图
plt.figure(figsize=(8, 8), dpi=300)
plt.pie(counts, labels=labels, autopct='%1.1f%%',
        colors=['#ff9999','#66b3ff','#99ff99','#ffcc99'],
        startangle=90)
plt.title('均值等级分布饼状图')
plt.savefig('studentScoresP_Mean_pie.png', dpi=300, bbox_inches='tight')
plt.close()

# 输出统计值
print(f"统计指标计算：\n最小值: {minValue:.2f}\n四分之一分位数: {quarterValue:.2f}\n中位数: {medianValue:.2f}\n四分之三分位数: {threeQuartersValue:.2f}\n最大值: {maxValue:.2f}")
print("\n等级分布统计：")
print(counts)

统计指标计算：
最小值: 62.33
四分之一分位数: 83.67
中位数: 90.33
四分之三分位数: 96.00
最大值: 114.67

等级分布统计：
等级
Poor         24
Moderate     22
Good         22
Excellent    22
Name: count, dtype: int64


In [7]:
# 任务6
# 重新读取文件studentScoresP_Mean.xlsx，分别可视化显示每位学生的平均成绩，要求包括图例、图标题，x、y轴均显示刻度且x轴刻度值以姓名显示，曲线颜色为红色，以png图片保存，分辨率为400dpi。此外，为显示美观，x轴姓名显示请选择倾斜显示，png图片命名为studentScoresP_Mean1.png。第2张图片x轴姓名显示选择间隔5名同学显示一次学生姓名，命名为studentScoresP_Mean2.png。

import pandas as pd
import matplotlib.pyplot as plt

# 读取Excel文件
df = pd.read_excel('studentScoresP_Mean.xlsx')

# 设置全局样式
plt.rcParams['font.sans-serif'] = ['SimHei']  # 解决中文显示问题
plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题

# ===== 第一张图：倾斜显示x轴姓名 =====
plt.figure(figsize=(16, 6), dpi=400)
plt.plot(df['姓名'], df['均值'], 'r-', label='平均成绩')
plt.title('学生平均成绩分布（全量显示）')
plt.xlabel('姓名', fontsize=10)
plt.ylabel('均值', fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=6)  # 倾斜45度并右对齐
plt.legend()
plt.grid(ls='--', alpha=0.5)
plt.tight_layout()  # 自动调整布局
plt.savefig('studentScoresP_Mean1.png', dpi=400)
plt.close()

# ===== 第二张图：间隔5名显示x轴姓名 =====
plt.figure(figsize=(16, 6), dpi=400)
plt.plot(df['姓名'], df['均值'], 'r-', label='平均成绩')
plt.title('学生平均成绩分布（间隔显示）')
plt.xlabel('姓名', fontsize=10)
plt.ylabel('均值', fontsize=10)

# 设置间隔显示的刻度
n = 5  # 间隔数量
plt.xticks(
    ticks=range(0, len(df), n),          # 位置：每5个索引
    labels=df['姓名'].iloc[::n].values,  # 标签：每5个姓名
    rotation=30,
    ha='right',
    fontsize=8
)
plt.legend()
plt.grid(ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig('studentScoresP_Mean2.png', dpi=400)
plt.close()

In [8]:
# 柱状图
import pandas as pd
import matplotlib.pyplot as plt

# 读取Excel文件
df = pd.read_excel('studentScoresP_Mean.xlsx')

# 设置全局样式
plt.rcParams['font.sans-serif'] = ['SimHei']  # 解决中文显示问题
plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题

# ===== 第一张图：倾斜显示x轴姓名 =====
plt.figure(figsize=(20, 8), dpi=400)
plt.bar(df['姓名'], df['均值'], color='red', label='平均成绩')
plt.title('学生平均成绩分布（全量显示）', fontsize=14)
plt.xlabel('姓名', fontsize=12)
plt.ylabel('均值', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)  # 倾斜45度并右对齐
plt.legend()
plt.grid(ls='--', alpha=0.5, axis='y')          # 仅显示y轴网格
plt.tight_layout()                              # 自动调整布局
plt.savefig('studentScoresP_Mean1.png', dpi=400)
plt.close()

# ===== 第二张图：间隔5名显示x轴姓名 =====
plt.figure(figsize=(20, 8), dpi=400)
plt.bar(df['姓名'], df['均值'], color='red', label='平均成绩')
plt.title('学生平均成绩分布（间隔显示）', fontsize=14)
plt.xlabel('姓名', fontsize=12)
plt.ylabel('均值', fontsize=12)

# 设置间隔显示的刻度
n = 5  # 间隔数量
ticks_pos = range(0, len(df), n)                # 刻度位置
labels = df['姓名'].iloc[::n].values             # 刻度标签
plt.xticks(
    ticks=ticks_pos,
    labels=labels,
    rotation=30,
    ha='right',
    fontsize=10
)
plt.legend()
plt.grid(ls='--', alpha=0.5, axis='y')
plt.tight_layout()
plt.savefig('studentScoresP_Mean2.png', dpi=400)
plt.close()